# Real-World MCP Example: OCI Usage MCP Server

This notebook demonstrates the client-side architecture for consuming a third-party MCP server.

Architecture:

`User → LangChain Agent → MCP Client → OCI Usage MCP Server → OCI APIs`

The key idea is that the application does **not** directly call OCI REST APIs. The MCP server owns that integration.

## Learning Objectives

- Understand real-world third-party MCP usage.
- Connect to an MCP server without implementing its server code.
- Discover MCP tools dynamically.
- Inspect tool names, descriptions, and schemas.
- Invoke an MCP tool.
- Connect MCP-discovered tools to a LangChain agent.
- Understand `stdio`, JSON-RPC, and the separation between MCP client and server.

## Important Note

The exact OCI Usage MCP package name, command-line arguments, authentication requirements, and tool schema may change over time. The configuration below is intentionally a template. Verify the current official OCI MCP documentation before running it in production.

The notebook demonstrates the architecture and client-side pattern.

In [ ]:
%pip install -U langchain langchain-openai langchain-mcp-adapters mcp python-dotenv

## Environment Variables

Keep API keys and credentials outside your source code.

Example `.env` file:

```text
OPENAI_API_KEY=your-openai-api-key
```

OCI authentication may be configured separately using the OCI SDK configuration or the authentication method required by the MCP server.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv('OPENAI_API_KEY'):
    print('OpenAI API key found.')
else:
    print('OpenAI API key not found.')

## Check for `uvx`

`uvx` can launch Python tools in an isolated environment. Conceptually, it is similar to `npx` in the Node.js ecosystem.

The MCP client can start the third-party MCP server as a subprocess, and the two processes can communicate through standard input/output.

In [ ]:
import shutil

if shutil.which('uvx'):
    print('uvx is installed.')
else:
    print('uvx was not found. Install uv before running the MCP server.')

## MCP Server Configuration

The client needs to know how to start the third-party server.

The exact package name below is a placeholder. Replace it with the current official OCI Usage MCP server package and arguments.

The important concept is:

```text
Our Python Application
        ↓
      uvx
        ↓
OCI Usage MCP Server
        ↓
    stdin/stdout
```

The local MCP server may then communicate with OCI services over HTTPS.

In [ ]:
MCP_SERVER_CONFIG = {
    'oci_usage': {
        'command': 'uvx',
        'args': [
            '<OCI_USAGE_MCP_SERVER_PACKAGE>'
        ],
        'transport': 'stdio',
    }
}

print(MCP_SERVER_CONFIG)

## Create the MCP Client

The client is responsible for communicating with the MCP server.

In a real-world setup, we usually do not need to inspect or modify the MCP server source code. We work through the protocol boundary.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(MCP_SERVER_CONFIG)

print('MCP client created.')

## Discover MCP Tools

The client asks the MCP server what tools it exposes.

Conceptually, this corresponds to the MCP `tools/list` operation.

The client does not hard-code the available tools. The server advertises them dynamically.

In [ ]:
tools = await client.get_tools()

print('Discovered tools:')
for tool in tools:
    print('-', tool.name)

## Inspect Discovered Tools

MCP tools expose structured metadata such as their name, description, and input schema.

The LLM can use this information to decide which tool to call.

In [ ]:
for tool in tools:
    print('=' * 60)
    print('Tool name:', tool.name)
    print('Description:', tool.description)
    print('Arguments:', tool.args)

## Find the OCI Usage Tool

The lesson describes a tool named `get_summarized_usage`. The actual name should always be verified from the discovered tool list because third-party server versions can change.

In [ ]:
usage_tool = next(
    (tool for tool in tools if tool.name == 'get_summarized_usage'),
    None
)

if usage_tool is None:
    print('get_summarized_usage was not discovered.')
    print('Review the discovered tools and update the tool name if necessary.')
else:
    print('Usage tool discovered:', usage_tool.name)
    print('Arguments:', usage_tool.args)

## Prepare a Usage Request

The exact input schema must be taken from `usage_tool.args`.

The example below follows the lesson's conceptual fields. Replace the placeholders with values matching the current server schema.

In [ ]:
usage_request = {
    'tenant_id': '<YOUR_OCI_TENANCY_ID>',
    'start_time': '<START_TIME>',
    'end_time': '<END_TIME>',
    'granularity': '<GRANULARITY>',
}

print(usage_request)

## Invoke the MCP Tool

This is the client-side equivalent of an MCP `tools/call` request.

The application sends the structured tool request to the MCP server. The MCP server performs the underlying OCI integration and returns the result.

In [ ]:
if usage_tool is not None:
    try:
        result = await usage_tool.ainvoke(usage_request)
        print('MCP tool result:')
        print(result)
    except Exception as exc:
        print('MCP tool invocation failed:')
        print(type(exc).__name__)
        print(str(exc))
else:
    print('Cannot invoke the tool because it was not discovered.')

## Connect MCP Tools to LangChain

The next step is to make the discovered MCP tools available to a LangChain agent.

The LLM sees the tool interface, decides whether a tool is required, and the MCP client/server architecture handles execution.

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='gpt-5.5',
    temperature=0,
)

print('Model initialized.')

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools,
)

print('LangChain agent created with MCP tools.')

## Ask the Agent a Question

The agent can now reason over the MCP-discovered tools.

For a production application, provide the required date range and OCI context explicitly, and ensure the MCP server has appropriate authentication and permissions.

In [ ]:
question = '''
Using the available OCI usage MCP tool, retrieve summarized usage
for the requested period and explain the result clearly.
'''

try:
    agent_result = await agent.ainvoke(
        {
            'messages': [
                {
                    'role': 'user',
                    'content': question,
                }
            ]
        }
    )

    print(agent_result)
except Exception as exc:
    print('Agent execution failed:')
    print(type(exc).__name__)
    print(str(exc))

## Architecture Comparison

### Before MCP

```text
Application
    ↓
LLM
    ↓
Local Python Tool
    ↓
OCI API
```

The application owns the integration.

### After MCP

```text
Application
    ↓
LLM / Agent
    ↓
MCP Client
    ↓  JSON-RPC over stdio
Third-Party MCP Server
    ↓  HTTPS / SDK
OCI API
```

The MCP server owns the OCI integration.

The application focuses on reasoning and orchestration.

## Why MCP Is Valuable in the Real World

MCP is especially useful when tools are:

- External
- Reusable
- Shared by multiple AI applications
- Maintained by another organization
- Connected to complex APIs

A vendor can build an MCP server once, and multiple compatible hosts can consume the tools.

The client does not need to duplicate the vendor's API integration logic.

## Debugging Third-Party MCP Servers

When you do not own the server, debug through the protocol boundary.

Check:

1. Did the server start successfully?
2. Did the MCP initialization handshake succeed?
3. Did `tools/list` return the expected tools?
4. Is the tool schema correct?
5. Are the arguments valid?
6. Does `tools/call` return a result?
7. Are authentication and permissions configured correctly?
8. Is the underlying OCI service available?

This is different from debugging a local Python function because the MCP server is an independent process and may be owned by another organization.

# Key Takeaways

1. In production, you usually consume an MCP server rather than build one.
2. Your application acts as the MCP client.
3. The client discovers tools dynamically.
4. MCP standardizes communication between clients and servers.
5. With stdio, the MCP client and server can run as separate local processes.
6. The MCP server can communicate with OCI over HTTPS.
7. Your application does not need to directly call OCI REST APIs.
8. The LLM sees tool descriptions and schemas, not the server's internal implementation.
9. MCP allows external tools to be reused by multiple AI hosts.
10. The biggest value of MCP appears when tools are external, complex, or shared.